In [16]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

In [28]:
df = pd.read_excel("2026.02.Fevrier-ExportAccor.xlsx")

In [6]:
df.shape

(3491, 20)

In [7]:
df.head(3)

,NOM BOUTIQUE,OPERATEUR,MACHINE,DATE,HEURE,STATUT,CODE EAN,NOM DU PRODUIT,QUANTITE,PRIX HT,VAT,PRIX TTC,TYPE,GAMME,MARQUE,FOURNISSEUR,ORDER ID (TICKET DE CAISSE),TEMPERATURE,METEO DU JOUR (MOYENNE),METEO DU MOIS (MOYENNE)
0,Novotel Paris Tour Eiffel,DIGITIZME,ACCESSORIES (armoire sèche),2026-02-01,00:11:56,DONE,NaN,ADAPTATEUR UNIVERSEL,1,NaN,19.97,14.0,NON-F&B,SOS,-,-,51847,NaN,NaN,NaN
1,Novotel Paris Tour Eiffel,DIGITIZME,ACCESSORIES (armoire sèche),2026-02-01,00:11:56,DONE,NaN,KIT DENTAIRE COLGATE,1,NaN,20.00,6.0,NON-F&B,SOS,COLGATE,ASTORE,51847,NaN,NaN,NaN
2,Mercure Paris Montmartre Sacré-Cœur,SELFLYSTORE,FRIGO MONTMARTRE (APERO PREMIUM 2),2026-02-01,00:15:28,DONE,3.497915e+12,Chips (90g),1,NaN,5.50,5.0,F&B,FOOD SALEE,-,ASTORE,50790,NaN,NaN,NaN


In [29]:
ORDER_COL = "ORDER ID (TICKET DE CAISSE)"
PRODUCT_COL = "NOM DU PRODUIT"
df = (
    df[df["STATUT"].str.upper() == "DONE"][[ORDER_COL, PRODUCT_COL]]
    .dropna()
    .drop_duplicates(subset = [ORDER_COL, PRODUCT_COL])
)


In [33]:
basket = (
    df.assign(value=1)
    .pivot_table(
        index=ORDER_COL,
        columns=PRODUCT_COL,
        values="value",
        aggfunc="max",
        fill_value=0
    )          
    .astype(bool)
)

In [34]:
basket

NOM DU PRODUIT,ADAPTATEUR UNIVERSEL,Adaptateur Novotel,Adaptateur universel Mercure,Amandes Truffées,Assortiment de Guimauve,BLAST SNACK CHOCOLAT NOIR SEL DE MER AMANDES,BLAST SNACK FIGUES GINGEMBRE AMANDES,BOUNTY 57g,BURGER BISTROT BŒUF CHAROLAIS,"Barre chocolatée (ex : Mars, Twix, Lion...)",...,Trousse de toilette Femme,Trousse de toilette Homme,Tuc (75g),Twix,VITAO GREEN TEA,VITTEL 50cl,Vittel en verre (50cl),WRAP VEGGIE ŒUF PARMESAN TOMATES,YAOURT NATURE 125G,lipton Ice tea
ORDER ID (TICKET DE CAISSE),,,,,,,,,,,,,,,,,,,,,
50478,False,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
50479,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
50480,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
50481,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
50482,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53745,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
53746,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
53747,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [38]:
frequent_itemsets = apriori(
    basket,
    min_support=0.01,   # à ajuster selon ton volume
    use_colnames=True
)

In [37]:
frequent_itemsets.shape

(47, 2)

In [43]:
rules = association_rules(
    frequent_itemsets,
    metric="lift",
    min_threshold=1.0
)

In [44]:
rules.shape

(6, 14)

In [45]:
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(VITTEL 50cl),(COCA COLA 33CL),0.168790,0.057325,0.012208,0.072327,1.261705,1.0,0.002532,1.016172,0.249542,0.057072,0.015914,0.142645
1,(COCA COLA 33CL),(VITTEL 50cl),0.057325,0.168790,0.012208,0.212963,1.261705,1.0,0.002532,1.056126,0.220035,0.057072,0.053143,0.142645
2,(VITTEL 50cl),(COCA COLA ZERO 33CL),0.168790,0.059979,0.011146,0.066038,1.101019,1.0,0.001023,1.006487,0.110381,0.051220,0.006446,0.125939
3,(COCA COLA ZERO 33CL),(VITTEL 50cl),0.059979,0.168790,0.011146,0.185841,1.101019,1.0,0.001023,1.020943,0.097604,0.051220,0.020513,0.125939
4,(SAN PELLEGRINO 50cl),(VITTEL 50cl),0.056263,0.168790,0.012208,0.216981,1.285511,1.0,0.002711,1.061546,0.235340,0.057357,0.057977,0.144654
5,(VITTEL 50cl),(SAN PELLEGRINO 50cl),0.168790,0.056263,0.012208,0.072327,1.285511,1.0,0.002711,1.017316,0.267200,0.057357,0.017021,0.144654


In [50]:
rules[
    (rules["confidence"] >= 0.20) &
    (rules["lift"] >= 1.20)
].sort_values(
    by=["lift", "confidence", "support"],
    ascending=False
)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
4,(SAN PELLEGRINO 50cl),(VITTEL 50cl),0.056263,0.16879,0.012208,0.216981,1.285511,1.0,0.002711,1.061546,0.235340,0.057357,0.057977,0.144654
1,(COCA COLA 33CL),(VITTEL 50cl),0.057325,0.16879,0.012208,0.212963,1.261705,1.0,0.002532,1.056126,0.220035,0.057072,0.053143,0.142645
